In [ ]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

## Docker Security

We want a 1-node Docker with 2 cores, 8GB of memory. 

In [ ]:
# Get the resources helper
resources = fablib.get_resources()
resources.update()

# if you have multiple nodes and want adequate resources, you need find a suitable site
nodesReq = 1
coresReq = 2
ramReq = 4

# we scale up the requirements a bit to account for the potential of others joining the selected site. 
totalCoreAvail = nodesReq * coresReq * 1.2
totalRamAvail = nodesReq * ramReq * 1.2

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(usableSite)

To avoid the scenario where all students joined the same site, the site selection is now random!

In [ ]:
usableSite = ['SRI','TOKY','BRIST'] # Current version of Docker rootless kit only works with inherent IPv4 sites. 
siteName = random.choice(usableSite)
#siteName = "CLEM"
sliceName = "Docker_NotRoot"
print(siteName)

network_name = 'ramnet'

slice = fablib.new_slice(name=sliceName)

# Network
net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(name=f"node{i}", 
                          site=siteName,
                          cores=coresReq,
                          ram=ramReq,
                          disk=30, 
                          image='default_ubuntu_24')
    iface = node.add_component(model='NIC_Basic', name='nic').get_interfaces()[0]
    iface.set_mode('config')
    net.add_interface(iface)

slice.submit()   
slice.wait_ssh()

In [ ]:
from ipaddress import IPv4Network

for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)

    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24")
    )

In [ ]:
import time

while True:
    time.sleep(10)
    slice.update()
    print("Slice state:", slice.get_state())
    print("Slice stable:", slice.isStable())
    if node.get_management_ip() != None:
        for node in slice.get_nodes():
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

## Create Inventory File

This is a simple inventory file that reuse the `node` variable from the earlier cell directly (fewer calls to `slice`). 

Programmatically design the generation of `inventory.yml` based on this information. 

In [ ]:
node = slice.get_node(name="node1")

docker_python, stderr = node.execute("which python3", quiet=True, output_file=f"{node.get_name()}-python.log");
inventory = f"""all:
  children:
    dockers:
      hosts:
        docker:
          ansible_host: "{node.get_management_ip()}"
          ansible_user: "{node.get_username()}"
          ansible_ssh_private_key_file: "{fablib.get_default_slice_key()["slice_private_key_file"]}"
          ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
          ansible_python_interpreter: "{docker_python.strip()}"
"""

from pathlib import Path
Path("playbook/inventory.yml").write_text(inventory)

## Playbook for Docker

- Primarily a declarative interpretation of [Docker's installation instruction](https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository)
    - This playbook is different from the Docker Swarm playbook in that the registry is not enabled. 
- A few extra steps to enable IPv6 support for Docker (FABRIC related)

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-docker.yml

In [ ]:
for node in slice.get_nodes():
    print(f"==== {node.get_name()} ====")
    stdout, stderr = node.execute("docker version", quiet=True);
    print(stdout)

In [ ]:
slice.delete()